In [ ]:
import uproot
import os
import dask_awkward as dak
import awkward as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from dask.diagnostics import ProgressBar
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
import joblib

import zfit
from zfit.loss import UnbinnedNLL
from zfit.minimize import Minuit
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mplhep as hep

In [ ]:
plt.style.use(hep.style.CMS)

In [ ]:
tree_path = "Events"
data_files = {f: tree_path for f in [
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022C.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022D.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022E.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022F.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022G.root"
]}

mc_files = {f: tree_path for f in [
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_mc_signal_2022.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_mc_signal_2022EE.root"
]}

In [ ]:
def get_b0_truth_mask(df):
    # --- 1. Constantes PDG ---
    PDG_B0        = 511
    PDG_KSTAR     = 313
    PDG_KAON_plus = 321
    PDG_PION_neg  = -211
    PDG_MUON_plus = -13
    PDG_MUON_neg  = 13
    PDG_PHOTON    = 22

    mu1_idx  = df["BPH_1Muon_genPartIdx"]
    mu2_idx  = df["BPH_2Muon_genPartIdx"]
    trk1_idx = df["Trk1_genPartIdx"]
    trk2_idx = df["Trk2_genPartIdx"]

    # Verificação de Identidade (PDG ID)
    mu1_pdg  = df["BPHGenPart_pdgId"][dak.mask(mu1_idx, mu1_idx >= 0)]
    mu2_pdg  = df["BPHGenPart_pdgId"][dak.mask(mu2_idx, mu2_idx >= 0)]
    trk1_pdg = df["BPHGenPart_pdgId"][dak.mask(trk1_idx, trk1_idx >= 0)]
    trk2_pdg = df["BPHGenPart_pdgId"][dak.mask(trk2_idx, trk2_idx >= 0)]

    flavor_match = (trk1_pdg == PDG_KAON_plus) & (trk2_pdg == PDG_PION_neg) & \
                   (mu1_pdg == PDG_MUON_plus) & (mu2_pdg == PDG_MUON_neg)

    # Linhagem do Hadron (K, Pi -> K* -> B0)
    mom_trk1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(trk1_idx, trk1_idx >= 0)]
    mom_trk2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(trk2_idx, trk2_idx >= 0)]
    
    mom_trk1_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_trk1_idx, mom_trk1_idx >= 0)]
    mom_trk2_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_trk2_idx, mom_trk2_idx >= 0)]
    
    # K e Pi devem vir do mesmo objeto K*0 
    match_kstar = (mom_trk1_idx == mom_trk2_idx) & (mom_trk1_idx >= 0) & \
                  (mom_trk1_pdg == PDG_KSTAR) & (mom_trk2_pdg == PDG_KSTAR)
    
    # O K*0 deve vir de um B0 (Verificando para ambos os traços)
    gmom_trk1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_trk1_idx, mom_trk1_idx >= 0)]
    gmom_trk2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_trk2_idx, mom_trk2_idx >= 0)]
    
    gmom_trk1_pdg = df["BPHGenPart_pdgId"][dak.mask(gmom_trk1_idx, gmom_trk1_idx >= 0)]
    gmom_trk2_pdg = df["BPHGenPart_pdgId"][dak.mask(gmom_trk2_idx, gmom_trk2_idx >= 0)]
    
    match_kstar_to_b0 = (gmom_trk1_idx == gmom_trk2_idx) & (gmom_trk1_idx >= 0) & \
                        (gmom_trk1_pdg == PDG_B0) & (gmom_trk2_pdg == PDG_B0)

    mom_mu1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mu1_idx, mu1_idx >= 0)]
    mom_mu2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mu2_idx, mu2_idx >= 0)]
    mom_mu1_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_mu1_idx, mom_mu1_idx >= 0)]
    mom_mu2_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_mu2_idx, mom_mu2_idx >= 0)]
    
    gmom_mu1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_mu1_idx, mom_mu1_idx >= 0)]
    gmom_mu2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_mu2_idx, mom_mu2_idx >= 0)]

    # Muon 1: Mãe é o B0 do hadron OU (Mãe é fóton e Avô é o B0 do hadron)
    mu1_from_same_b0 = (mom_mu1_idx == gmom_trk1_idx) | \
                       ((mom_mu1_pdg == PDG_PHOTON) & (gmom_mu1_idx == gmom_trk1_idx))

    mu2_from_same_b0 = (mom_mu2_idx == gmom_trk1_idx) | \
                       ((mom_mu2_pdg == PDG_PHOTON) & (gmom_mu2_idx == gmom_trk1_idx))

    # --- Máscara Final ---
    full_mask = flavor_match & match_kstar & match_kstar_to_b0 & \
                mu1_from_same_b0 & mu2_from_same_b0
    
    return ak.fill_none(full_mask, False)

In [ ]:
def add_derived_columns(df):
    df["BToTrkTrkMuMu_l_xy_sig"] = df['BToTrkTrkMuMu_l_xy'] / df['BToTrkTrkMuMu_l_xy_unc']
    df["BToTrkTrkMuMu_dca_sig"]  = df['BToTrkTrkMuMu_dca'] / df['BToTrkTrkMuMu_dcaErr']
    df["BToTrkTrkMuMu_trk1_dca_sig"] = df['BToTrkTrkMuMu_trk1_dca'] / df['BToTrkTrkMuMu_trk1_dcaErr']
    df["BToTrkTrkMuMu_trk2_dca_sig"] = df['BToTrkTrkMuMu_trk2_dca'] / df['BToTrkTrkMuMu_trk2_dcaErr']
    return df

In [ ]:
def apply_selection(df, mode="mc"):

    kstar_pdg = 0.892
    mask_kpi_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) <= 0.150
    mask_pik_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg) <= 0.150
    is_kpi_closer = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) < \
                    abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg)
    
    mask = (
        # Filtro de massa de dimuons (regiões de sinal, excluindo J/psi e Psi2S)
        (((df['MuMu_mass'] > 1.0) & (df['MuMu_mass'] < 2.7)) | 
         ((df['MuMu_mass'] > 4.0) & (df['MuMu_mass'] < 6.0))) & 
        
        # Pelo menos uma das combinações deve estar na janela de 3 sigma
        (mask_kpi_window | mask_pik_window) & 
        
        # Aplicamos a lógica de decisão de sabor (escolhemos a hipótese Kpi se ela for a melhor)
        (is_kpi_closer) &
        
        # Trigger e cortes cinemáticos dos múons
        (df['HLT_DoubleMu4_3_LowMass'] == 1) & 
        (df['BPH_1Muon_pt'] > 4.0) & (df['BPH_2Muon_pt'] > 4.0) & 
        (abs(df['BPH_1Muon_eta']) < 2.4) & (abs(df['BPH_2Muon_eta']) < 2.4)
    )
    if mode == "mc":
        mask = mask & (df['BToTrkTrkMuMu_fit_mass_Kpi'] >= 5.133542769) & (df['BToTrkTrkMuMu_fit_mass_Kpi'] <= 5.416657231)
        mask = mask & get_b0_truth_mask(df) # No MC junta a mask de thruth Matching !!!
    else:
        mask = mask & (
            ((df['BToTrkTrkMuMu_fit_mass_Kpi'] > 5.0) & (df['BToTrkTrkMuMu_fit_mass_Kpi'] < 5.133542769)) | 
            ((df['BToTrkTrkMuMu_fit_mass_Kpi'] > 5.416657231) & (df['BToTrkTrkMuMu_fit_mass_Kpi'] < 5.6))
        )
    
    cols_to_keep = [
        'BToTrkTrkMuMu_l_xy_sig', 'BToTrkTrkMuMu_dca_sig', 
        'BToTrkTrkMuMu_trk1_dca_sig', 'BToTrkTrkMuMu_trk2_dca_sig', 
        'BToTrkTrkMuMu_fit_ditrack_mass_Kpi', 'BToTrkTrkMuMu_fit_pt', 
        'BToTrkTrkMuMu_fit_cos2D', 'BToTrkTrkMuMu_svprob', 'event', "BToTrkTrkMuMu_fit_mass_Kpi"
    ]
    return df[cols_to_keep][mask]

In [ ]:
def build_dataframe(file_dict, label="Dataset", mode="mc"):
    print(f"\nProcessando {label}...")
    
    df = uproot.dask(file_dict)
    df = add_derived_columns(df)
    df = apply_selection(df, mode=mode)
    
    with ProgressBar():
        awkward_array = df.compute()
        df_pandas = ak.to_dataframe(awkward_array).reset_index(drop=True)
    
    print(f"{label} finalizado. Linhas: {len(df_pandas)}")
    return df_pandas

In [ ]:
df_data = build_dataframe(data_files, "Data (Sidebands)", mode="data")
df_mc   = build_dataframe(mc_files, "MC Signal (Truth Matched)", mode="mc")

In [ ]:
df_mc['target'] = 1
df_data['target'] = 0

In [ ]:
df_total = pd.concat([df_mc, df_data], ignore_index=True)

In [ ]:
df_total.replace([np.inf, -np.inf], np.nan, inplace=True)
df_total.dropna(inplace=True)

In [ ]:
df_total['fold_id'] = df_total['event'] % 11

In [ ]:
df_0 = df_total[df_total['fold_id'] == 0]

In [ ]:
model = joblib.load("bdt_models_final/xgboost_fold_0.joblib")

In [ ]:
probabilities = model.predict_proba(df_0.drop(columns=["target", "BToTrkTrkMuMu_fit_mass_Kpi", "fold_id", "event"]))[:, 1]
df_0['bdt_score'] = probabilities

In [ ]:
df = df_0.copy()

In [ ]:
L_data_fb = 31.69
sigma_pb  = 17370000 
BR        = 6.30e-7
N_mc_gen  = 19906620 + 5542508
#N_mc_gen = 5542508 #usando eras C,D
            
L_data_pb = L_data_fb * 1000 
n_expected = L_data_pb * sigma_pb * BR
scale_factor = n_expected / N_mc_gen

print(f"Eventos esperados: {n_expected:.2f}")
print(f"Scale Factor: {scale_factor:.4f}")

var_name = 'BToTrkTrkMuMu_fit_mass_Kpi'
corte_bdt = 0.

mask_bkg = (df['target'] == 0) & (df['bdt_score'] > corte_bdt)
data_bkg = df[mask_bkg][var_name]

mask_sig = (df['target'] == 1) & (df['bdt_score'] > corte_bdt)
data_sig = df[mask_sig][var_name]

xmin, xmax = data_bkg.min(), data_bkg.max() 
bins = np.linspace(xmin, xmax, 60)
h_bkg, _ = np.histogram(data_bkg, bins=bins)

weights_sig = np.full(len(data_sig), 
                      scale_factor)
h_sig, _ = np.histogram(data_sig, bins=bins, weights=weights_sig)
fig, ax = plt.subplots(figsize=(10, 8))

hep.histplot(
    h_bkg,
    bins=bins,
    ax=ax,
    histtype='fill',
    color='gray',
    alpha=0.5,
    edgecolor='black',
    label=f'Background (Data {L_data_fb} fb$^{{-1}}$)'
)

hep.histplot(
    h_sig,
    bins=bins,
    ax=ax,
    histtype='step',
    linewidth=3,
    color='red',
    label=f'Signal (Normalized by Lum)\nBR={BR:.1e}'
)

#ax.set_xlabel(var_name.replace('_', ' '), fontsize=20)
ax.set_xlabel(r'$m(K^{+}\pi^{-}\mu^{+}\mu^{-})\;[\mathrm{GeV}]$')
ax.set_ylabel(f"Events / 0.1 bin (GeV)", fontsize=20)
ax.set_xlim(xmin, xmax)
ax.legend(loc='upper right', fontsize=16)
texts = hep.label.exp_label(
    exp="CMS",
    label="Work in Progress",
    data=True,
    lumi=L_data_fb,
    year=2022,
    com=13.6,
    ax=ax
)
texts[1].set_fontsize(25) 

plt.tight_layout()
plt.savefig("PlotsEE/Comparrison_dataSideband_mcSignal_lumiscale.png")
plt.show()

In [ ]:
mass_min_lsb, mass_max_lsb = 5.0, 5.133542769
mass_min_rsb, mass_max_rsb = 5.416657231, 5.6

obs_lsb = zfit.Space("BToTrkTrkMuMu_fit_mass_Kpi", limits=(mass_min_lsb, mass_max_lsb))
obs_rsb = zfit.Space("BToTrkTrkMuMu_fit_mass_Kpi", limits=(mass_min_rsb, mass_max_rsb))
obs_bkg = obs_lsb + obs_rsb

mask_sidebands = (df['target'] == 0) & (df['bdt_score'] > corte_bdt)
data_sidebands_np = df[mask_sidebands]['BToTrkTrkMuMu_fit_mass_Kpi'].to_numpy()

data_zfit = zfit.Data.from_numpy(obs=obs_bkg, array=data_sidebands_np)

In [ ]:
lambda_bkg = zfit.Parameter("lambda_bkg", -0.5, -10.0, 100.0)
bkg_model = zfit.pdf.Exponential(lambda_=lambda_bkg, obs=obs_bkg)

loss = UnbinnedNLL(model=bkg_model, data=data_zfit)
minimizer = Minuit()
result = minimizer.minimize(loss)

result.hesse()
print(f"Resultado do Fit:\n{result}")

In [ ]:
plt.figure(figsize=(10, 8))

nbins = 60
counts, bin_edges = np.histogram(data_sidebands_np, bins=nbins, range=(mass_min_lsb, mass_max_rsb))
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]

hep.histplot(counts, bins=bin_edges, yerr=True, color='black', histtype='errorbar', label='Data Sidebands')
x_plot = np.linspace(mass_min_lsb, mass_max_rsb, 1000)
pdf_values = bkg_model.pdf(x_plot).numpy()
yield_total = len(data_sidebands_np)
plt.plot(x_plot, pdf_values * yield_total * bin_width, color='red', lw=3, label='Background Fit (Exp)')

plt.axvspan(mass_max_lsb, mass_min_rsb, color='gray', alpha=0.2, label='Signal Region (Blinded)')
plt.xlabel(r'$m(K^{+}\pi^{-}\mu^{+}\mu^{-})\;[\mathrm{GeV}]$')
plt.ylabel(f"Events / {bin_width:.3f} GeV")
plt.legend(fontsize=14)
hep.cms.label(label="Preliminary", data=True, lumi=L_data_fb, year=2022, com=13.6)

plt.tight_layout()
plt.savefig("PlotsEE/Fit_Background_regions.png")
plt.show()

In [ ]:
def integral_exp(lmbda, a, b):
    if abs(lmbda) < 1e-12: 
        return b - a
    return (np.exp(lmbda * b) - np.exp(lmbda * a)) / lmbda

In [ ]:
mass_min_lsb, mass_max_lsb = 5.0, 5.133542769
mass_min_sig, mass_max_sig = 5.133542769, 5.416657231
mass_min_rsb, mass_max_rsb = 5.416657231, 5.6

obs_lsb = zfit.Space("BToTrkTrkMuMu_fit_mass_Kpi", limits=(mass_min_lsb, mass_max_lsb))
obs_rsb = zfit.Space("BToTrkTrkMuMu_fit_mass_Kpi", limits=(mass_min_rsb, mass_max_rsb))
obs_sidebands = obs_lsb + obs_rsb

obs_signal_region = zfit.Space("BToTrkTrkMuMu_fit_mass_Kpi", limits=(mass_min_sig, mass_max_sig))

In [ ]:
cuts = np.linspace(0, 1, 1000)
B_est_list = []
S_est_list = []
FoM_list = []

print("Iniciando BDT Scan...")

for cut_value in cuts:
    df_cut = df[df["bdt_score"] > cut_value]
    mask_bkg = (df_cut["target"] == 0) & (
        ((df_cut["BToTrkTrkMuMu_fit_mass_Kpi"] >= mass_min_lsb) & (df_cut["BToTrkTrkMuMu_fit_mass_Kpi"] <= mass_max_lsb)) |
        ((df_cut["BToTrkTrkMuMu_fit_mass_Kpi"] >= mass_min_rsb) & (df_cut["BToTrkTrkMuMu_fit_mass_Kpi"] <= mass_max_rsb))
    )
    mass_bkg = df_cut[mask_bkg]["BToTrkTrkMuMu_fit_mass_Kpi"].to_numpy()
    N_bkg_sb = len(mass_bkg)

    mask_sig = (df_cut["target"] == 1) & \
               (df_cut["BToTrkTrkMuMu_fit_mass_Kpi"] >= mass_min_sig) & \
               (df_cut["BToTrkTrkMuMu_fit_mass_Kpi"] <= mass_max_sig)
    mass_sig_mc = df_cut[mask_sig]["BToTrkTrkMuMu_fit_mass_Kpi"].to_numpy()    
    S_est = len(mass_sig_mc) * scale_factor

    if N_bkg_sb < 15:
        B_est_list.append(0.0)
        S_est_list.append(S_est)
        FoM_list.append(0.0)
        continue

    tau_param = zfit.Parameter(f"tau_cut_{cut_value:.3f}", -1.0, -20.0, 0.0)
    model_bkg = zfit.pdf.Exponential(lambda_=tau_param, obs=obs_sidebands)
    data_zfit = zfit.Data.from_numpy(obs=obs_sidebands, array=mass_bkg)
    
    loss = UnbinnedNLL(model=model_bkg, data=data_zfit)
    minimizer = Minuit()
    result = minimizer.minimize(loss)
    
    tau_val = float(tau_param.value())
    num = integral_exp(tau_val, mass_min_sig, mass_max_sig)
    den = integral_exp(tau_val, mass_min_lsb, mass_max_lsb) + \
          integral_exp(tau_val, mass_min_rsb, mass_max_rsb)
    
    frac = num / den
    B_est = N_bkg_sb * frac    
    fom = S_est / np.sqrt(S_est + B_est) if (S_est + B_est) > 0 else 0.0

    B_est_list.append(B_est)
    S_est_list.append(S_est)
    FoM_list.append(fom)
    
    if cut_value % 0.2 < 0.01:
        print(f"Cut: {cut_value:.2f} | S: {S_est:.1f} | B_est: {B_est:.1f} | Significância: {fom:.3f}")

print("\nScan finalizado com sucesso!")

In [ ]:
FoM_list = np.array(FoM_list)
best_idx = np.argmax(FoM_list)
best_cut = cuts[best_idx]
best_fom = FoM_list[best_idx]

print("\n" + "="*40)
print(f"OPTIMIZATION SUMMARY")
print(f"Best BDT Cut: {best_cut:.3f}")
print(f"Max Significance: {best_fom:.3f}")
print(f"S: {S_est_list[best_idx]:.2f} | B: {B_est_list[best_idx]:.2f}")
print("="*40)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(cuts, FoM_list, 'ko', markersize=4, label=r"Significance $S/\sqrt{S+B}$")
ax.axvline(best_cut, color='red', linestyle='--', linewidth=2, alpha=0.8, label=f"Best cut = {best_cut:.2f}")
ax.text(best_cut, best_fom * 1.05, f"({best_cut:.2f}, {best_fom:.3f})", 
        color='red', fontsize=18, fontweight='bold', ha='center', va='bottom')

ax.set_xlabel("BDT Score Cut", fontsize=24)
ax.set_ylabel(r"Significance $[S/\sqrt{S+B}]$", fontsize=24)
ax.set_ylim(0, max(FoM_list) * 1.4)
ax.set_xlim(0.96, 1.0)
ax.tick_params(axis='both', which='major', labelsize=20, direction='in', length=10)
ax.tick_params(axis='both', which='minor', direction='in', length=5)
ax.grid(False)

hep.cms.label(ax=ax, label="Work in Progress", data=True, lumi=L_data_fb , year="2022", com=13.6, fontsize=20)
ax.legend(loc='upper left', fontsize=18, frameon=False)

plt.tight_layout()
plt.savefig("PlotsEE/BDT_Optimization_Final.png", dpi=300)
plt.show()